# HireSignal: soft prompt tuning of the Matcher (Qwen3-4B-Instruct-2507), EXPERIMENTAL

Settings: **Accelerator GPU T4 x2 (or P100)**, **Internet ON** (base model and Matcher LoRA from Hugging Face, public).
Input: PRIVATE Kaggle dataset `hiresignal-soft-prompt` made from `hiresignal_soft_prompt_bundle.zip`
(`python -m prompt_tuning.make_bundle`). The locked test split is not in the bundle.

Four arms, all through the SAME backend (transformers, one GPU, same dtype, greedy, max 192 new tokens, prompt p1,
same 202 val prompts with the same BM25 refs, same validator and metrics):
`base`, `lora` (Matcher LoRA merged), `soft_prompt` (trained on frozen base), `lora_soft_prompt` (trained on the frozen merged LoRA model).

Training uses `finetune/data/sft_train.jsonl` only. Val (`sft_val.jsonl`) gives the val loss used to pick the checkpoint.
Every cell is resumable: re-run it after an interruption. Estimated times (not measured) are in `prompt_tuning/README.md`.

In [ ]:
!pip install -q "transformers==5.19.0" "peft==0.21.2" "accelerate==1.15.0" "pydantic==2.14.1" "safetensors" "psutil"
import os, shutil
src = '/kaggle/input/hiresignal-soft-prompt'
zips = [os.path.join(r, f) for r, _, fs in os.walk(src) for f in fs if f.endswith('.zip')]
if zips:
    shutil.unpack_archive(zips[0], '/kaggle/working/hs')
else:
    shutil.copytree(src, '/kaggle/working/hs', dirs_exist_ok=True)
os.chdir('/kaggle/working/hs'); print(sorted(os.listdir('.')))

In [ ]:
import json, hashlib, torch
m = json.load(open('prompt_tuning/data/manifest.json'))
for f, info in m['files'].items():
    h = hashlib.sha256(open(f, 'rb').read()).hexdigest()
    print(f, 'OK' if h == info['sha256'] else 'SHA MISMATCH  <- stop here', info.get('use'))
print(json.dumps(m['leakage_audit'], indent=1))
assert not os.path.exists('benchmark/data/test.jsonl'), 'test split must not be in the bundle'
print(torch.__version__, torch.cuda.get_device_name(0))
DTYPE = 'bfloat16' if torch.cuda.is_bf16_supported(including_emulation=False) else 'float16'  # T4/P100 -> float16
print('dtype for ALL arms:', DTYPE)
os.environ['DTYPE'] = DTYPE
os.environ['LORA'] = 'HireSignal-5DS-Team/hiresignal-qwen3-4b-matcher-lora'; os.environ['LREV'] = '0b6925b1e3c89b6b266b10bcd30af4c6a0c2dc3a'
OUT = '/kaggle/working/results'; os.environ['OUT'] = OUT

## 1. Reference arms (no training): base and Matcher LoRA (merged), same backend

In [ ]:
!python -m prompt_tuning.evaluate --arm base --device cuda --dtype $DTYPE --run-id base-$DTYPE-val --out-root $OUT
!python -m prompt_tuning.evaluate --arm lora --lora $LORA --lora-revision $LREV --device cuda --dtype $DTYPE --run-id lora-$DTYPE-val --out-root $OUT

## 2. Soft prompt on the frozen base (32 virtual tokens, text init, lr 3e-2, 3 epochs, select on val loss)

In [ ]:
!python -m prompt_tuning.train_soft_prompt --run-id sp-base-v1 --runs-dir /kaggle/working/runs --device cuda --dtype $DTYPE --grad-ckpt

## 3. Soft prompt on the frozen, merged Matcher LoRA

In [ ]:
!python -m prompt_tuning.train_soft_prompt --run-id sp-lora-v1 --runs-dir /kaggle/working/runs --lora $LORA --lora-revision $LREV --device cuda --dtype $DTYPE --grad-ckpt

## 4. Soft prompt arms on the same 202 val requirements

In [ ]:
!python -m prompt_tuning.evaluate --arm soft_prompt --soft-prompt /kaggle/working/runs/sp-base-v1/soft_prompt --device cuda --dtype $DTYPE --run-id sp-base-v1-$DTYPE-val --out-root $OUT
!python -m prompt_tuning.evaluate --arm lora_soft_prompt --lora $LORA --lora-revision $LREV --soft-prompt /kaggle/working/runs/sp-lora-v1/soft_prompt --device cuda --dtype $DTYPE --run-id sp-lora-v1-$DTYPE-val --out-root $OUT

## 5. Comparison (paired McNemar + bootstrap CI vs base)

In [ ]:
!python -m prompt_tuning.compare --out-root $OUT --runs base=base-$DTYPE-val lora=lora-$DTYPE-val soft_prompt=sp-base-v1-$DTYPE-val lora_soft_prompt=sp-lora-v1-$DTYPE-val

## 6. Save outputs
Download `/kaggle/working/soft_prompt_results.zip` (eval run dirs + COMPARISON) and `soft_prompt_runs.zip`
(soft prompts, ~330 KB each, loss logs, provenance; checkpoints excluded). Commit them under
`benchmark/results/experimental/soft_prompt/` on `session/s3-matcher-soft-prompt`. Do not run on the test split.

In [ ]:
import shutil, glob
for d in glob.glob('/kaggle/working/runs/*/checkpoint-*'):
    shutil.rmtree(d)
shutil.make_archive('/kaggle/working/soft_prompt_results', 'zip', '/kaggle/working', 'results')
shutil.make_archive('/kaggle/working/soft_prompt_runs', 'zip', '/kaggle/working', 'runs')
print('done')